# V3 MNLI QLoRA Training Notebook

This is a Colab-friendly notebook for training an OSS NLI signal.

Design choices in this version:
- defaults to a Colab-safer Llama-family model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`)
- supports switching to `mistralai/Mistral-7B-v0.1` on a stronger GPU

Label mapping:
- `entailment` -> `SUPPORT`
- `contradiction` -> `CONTRADICT`
- `neutral` -> `NOT_ENOUGH_INFO`


In [ ]:
%%capture
!pip install -U transformers datasets accelerate peft bitsandbytes sentencepiece scipy

In [ ]:
import os
from dataclasses import dataclass, field
from pathlib import Path

import torch
from datasets import DatasetDict, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
@dataclass(slots=True)
class DatasetConfig:
    hf_dataset_name: str = 'multi_nli'
    train_split: str = 'train'
    eval_split: str = 'validation_matched'
    max_train_examples: int | None = 5000
    max_eval_examples: int | None = 1000
    seed: int = 42


@dataclass(slots=True)
class TrainingConfig:
    model_name: str = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
    output_dir: str = '/content/v3_outputs/tinyllama-mnli-qlora'
    logging_dir: str = '/content/v3_logs/tinyllama-mnli-qlora'
    max_seq_length: int = 256
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    learning_rate: float = 2e-4
    num_train_epochs: float = 1.0
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    lr_scheduler_type: str = 'cosine'
    logging_steps: int = 10
    save_steps: int = 100
    eval_steps: int = 100
    save_total_limit: int = 2
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple[str, ...] = (
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    )
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = 'nf4'
    bnb_4bit_compute_dtype: str = 'float16'
    bnb_4bit_use_double_quant: bool = True
    report_to: str = 'none'


@dataclass(slots=True)
class NotebookConfig:
    mount_drive: bool = False
    drive_output_root: str = '/content/drive/MyDrive/v3_nli_training'
    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)


config = NotebookConfig()

if config.mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    config.training.output_dir = f"{config.drive_output_root}/outputs"
    config.training.logging_dir = f"{config.drive_output_root}/logs"

from pathlib import Path
Path(config.training.output_dir).mkdir(parents=True, exist_ok=True)
Path(config.training.logging_dir).mkdir(parents=True, exist_ok=True)

print(config)

In [ ]:
MNLI_TO_PROJECT_LABEL = {
    'entailment': 'SUPPORT',
    'contradiction': 'CONTRADICT',
    'neutral': 'NOT_ENOUGH_INFO',
}


class MNLIDatasetPreparer:
    def __init__(self, dataset_config: DatasetConfig):
        self.config = dataset_config

    def prepare(self) -> DatasetDict:
        print('[dataset] Loading MNLI from Hugging Face datasets')
        raw = load_dataset(self.config.hf_dataset_name)

        train = raw[self.config.train_split]
        eval_ds = raw[self.config.eval_split]

        if self.config.max_train_examples is not None:
            train = train.shuffle(seed=self.config.seed).select(range(self.config.max_train_examples))
        if self.config.max_eval_examples is not None:
            eval_ds = eval_ds.shuffle(seed=self.config.seed).select(range(self.config.max_eval_examples))

        train = train.filter(lambda x: x['label'] in [0, 1, 2])
        eval_ds = eval_ds.filter(lambda x: x['label'] in [0, 1, 2])

        train = train.map(self._format_example, remove_columns=train.column_names)
        eval_ds = eval_ds.map(self._format_example, remove_columns=eval_ds.column_names)

        print(f"[dataset] train size: {len(train)}")
        print(f"[dataset] eval size : {len(eval_ds)}")
        return DatasetDict({'train': train, 'eval': eval_ds})

    def _format_example(self, example: dict) -> dict:
        id_to_label = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
        original_label = id_to_label[example['label']]
        label = MNLI_TO_PROJECT_LABEL[original_label]

        prompt = (
            'You are performing natural language inference for open-domain claim verification.\n'
            'Given a premise and a claim, output exactly one label:\n'
            'SUPPORT\n'
            'CONTRADICT\n'
            'NOT_ENOUGH_INFO'
        )
        input_text = (
            f"Premise: {example['premise'].strip()}\n"
            f"Claim: {example['hypothesis'].strip()}\n"
            'Label:'
        )
        prompt_text = f"{prompt}\n\n{input_text}"

        return {
            'prompt': prompt_text,
            'completion': f" {label}",
            'input_text': input_text,
            'label_text': label,
            'text': f"{prompt_text} {label}",
        }


class VerboseProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        pieces = [f"{k}={v}" for k, v in sorted(logs.items())]
        print(f"[train] step={state.global_step} " + ' '.join(pieces))


class QLoRATrainer:
    def __init__(self, training_config: TrainingConfig):
        self.config = training_config

    def train(self, dataset: DatasetDict):
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA GPU is required for this notebook.')

        torch.cuda.empty_cache()

        print(f"[trainer] Loading tokenizer: {self.config.model_name}")
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_name,
            use_fast=True,
            trust_remote_code=False,
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = 'right'

        print(f"[trainer] Loading 4-bit model: {self.config.model_name}")
        bf16_enabled = torch.cuda.is_bf16_supported()
        fp16_enabled = not bf16_enabled
        compute_dtype = torch.bfloat16 if bf16_enabled else torch.float16
        quant_config = BitsAndBytesConfig(
            load_in_4bit=self.config.load_in_4bit,
            bnb_4bit_quant_type=self.config.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=self.config.bnb_4bit_use_double_quant,
        )
        model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            quantization_config=quant_config,
            device_map={'': 0},
            trust_remote_code=False,
        )
        # Fix #3: sync pad_token_id so the generation config doesn't treat the
        # pad/EOS token as a premature stop signal during inference.
        model.config.pad_token_id = tokenizer.pad_token_id
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(model)
        # use_reentrant=False avoids a FutureWarning and prevents NaN gradients with QLoRA
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

        peft_config = LoraConfig(
            r=self.config.lora_r,
            lora_alpha=self.config.lora_alpha,
            lora_dropout=self.config.lora_dropout,
            bias='none',
            task_type='CAUSAL_LM',
            target_modules=list(self.config.target_modules),
        )
        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()

        def tokenize_fn(batch: dict) -> dict:
            encoded = tokenizer(
                batch['text'],
                truncation=True,
                max_length=self.config.max_seq_length,
                padding=False,
            )
            prompt_encoded = tokenizer(
                batch['prompt'],
                truncation=True,
                max_length=self.config.max_seq_length,
                padding=False,
            )
            labels = []
            for full_ids, prompt_ids in zip(encoded['input_ids'], prompt_encoded['input_ids']):
                label_ids = full_ids.copy()
                # Fix #1: clamp so at least one completion token stays unmasked.
                # Without this, examples where the prompt fills the full window
                # get every position set to -100, producing silent zero loss.
                n_mask = min(len(prompt_ids), len(full_ids) - 1)
                label_ids[:n_mask] = [-100] * n_mask
                labels.append(label_ids)
            encoded['labels'] = labels
            return encoded

        print('[trainer] Tokenizing datasets')
        train_dataset = dataset['train'].map(
            tokenize_fn,
            batched=True,
            remove_columns=dataset['train'].column_names,
            desc='Tokenizing train split',
        )
        eval_dataset = dataset['eval'].map(
            tokenize_fn,
            batched=True,
            remove_columns=dataset['eval'].column_names,
            desc='Tokenizing eval split',
        )

        training_args = TrainingArguments(
            output_dir=self.config.output_dir,
            logging_dir=self.config.logging_dir,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_train_epochs,
            warmup_ratio=self.config.warmup_ratio,
            weight_decay=self.config.weight_decay,
            lr_scheduler_type=self.config.lr_scheduler_type,
            bf16=bf16_enabled,
            fp16=fp16_enabled,
            logging_steps=self.config.logging_steps,
            eval_strategy='steps',
            eval_steps=self.config.eval_steps,
            save_strategy='steps',
            save_steps=self.config.save_steps,
            save_total_limit=self.config.save_total_limit,
            report_to=self.config.report_to,
            remove_unused_columns=False,
            dataloader_num_workers=0,
            run_name='colab-qlora-nli',
            disable_tqdm=False,
            label_names=['labels'],
        )

        collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=collator,
            processing_class=tokenizer,
            callbacks=[VerboseProgressCallback()],
        )

        print('[trainer] Starting training')
        trainer.train()

        final_dir = os.path.join(self.config.output_dir, 'final_adapter')
        print(f'[trainer] Saving final adapter to {final_dir}')
        trainer.save_model(final_dir)
        tokenizer.save_pretrained(final_dir)
        return trainer, tokenizer


In [ ]:
print('=' * 80)
print('PREPARING DATASET')
print('=' * 80)
preparer = MNLIDatasetPreparer(config.dataset)
dataset = preparer.prepare()
dataset['train'][0]

In [ ]:
print('=' * 80)
print('STARTING QLORA TRAINING')
print('=' * 80)
trainer_wrapper = QLoRATrainer(config.training)
trainer, tokenizer = trainer_wrapper.train(dataset)

In [ ]:
example_prompt = (
    'You are performing natural language inference for open-domain claim verification.\n'
    'Given a premise and a claim, output exactly one label:\n'
    'SUPPORT\n'
    'CONTRADICT\n'
    'NOT_ENOUGH_INFO\n\n'
    'Premise: A soccer game with multiple males playing.\n'
    'Claim: Some men are playing a sport.\n'
    'Label:'
)

inputs = tokenizer(example_prompt, return_tensors='pt').to(trainer.model.device)
with torch.no_grad():
    outputs = trainer.model.generate(**inputs, max_new_tokens=5)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
save_dir = "/content/drive/MyDrive/v3_nli_training/final_adapter"                                                                             
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)                                                                                                           
print(f"Saved to {save_dir}")                                   
